<a href="https://colab.research.google.com/github/SabrinaZ600/AI-four-dimension-project/blob/main/Product_Strength_Sony.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformers sentence-transformers -q
!pip install pandas tqdm matplotlib -q

In [12]:
import pandas as pd

import numpy as np

from tqdm import tqdm

import torch
import re

from transformers import pipeline

In [3]:
from google.colab import files

uploaded = files.upload()

df = pd.read_csv(next(iter(uploaded)))

Saving Sony.csv to Sony.csv


In [13]:
df["标题"]=df["标题"].fillna("")

df["内容"]=df["内容"].fillna("")

df["full_text"]=df["标题"]+" "+df["内容"]

df=df.drop_duplicates("full_text")

print(df.shape)

(100, 23)


In [14]:
PRODUCT_KEYWORDS = [

"耳机",

"降噪",

"ANC",

"XM3",

"XM4",

"XM5",

"WH-1000",

"WF-1000",

"头戴",

"入耳",

"蓝牙",

"佩戴",

"音质",

"续航",

"耳罩",

"主动降噪"

]

In [15]:
def is_product_post(text):

    text = str(text)

    return any(k in text for k in PRODUCT_KEYWORDS)

In [16]:
df["is_product"] = df["full_text"].apply(is_product_post)

df = df[df["is_product"]]

print(df.shape)

(87, 24)


In [17]:
PRODUCT_TOPICS = {

"ANC":[
"降噪",
"主动降噪",
"ANC",
"通透模式",
"环境音"
],

"Sound":[
"音质",
"声音",
"低音",
"高音",
"解析",
"声场",
"人声"
],

"Comfort":[
"佩戴",
"舒服",
"舒适",
"耳压",
"夹头",
"耳罩"
],

"Battery":[
"续航",
"电池",
"充电",
"快充"
],

"Design":[
"外观",
"设计",
"颜值",
"做工"
],

"Connectivity":[
"蓝牙",
"连接",
"断连",
"稳定",
"延迟"
],

"Price":[
"价格",
"性价比",
"贵",
"便宜",
"优惠"

]

}

In [18]:
def split_sentences(text):

    text = str(text)

    s = re.split("[。！？；\n]", text)

    s = [x.strip() for x in s if len(x.strip())>3]

    return s

In [19]:
example="""

XM5降噪很好。

戴着很舒服。

就是太贵。

"""

split_sentences(example)

['XM5降噪很好', '戴着很舒服', '就是太贵']

In [20]:
def detect_topics(sentence):

    result=[]

    for topic, words in PRODUCT_TOPICS.items():

        if any(w in sentence for w in words):

            result.append(topic)

    return result

In [21]:
detect_topics("XM5降噪很好")

['ANC']

In [22]:
classifier = pipeline(

"sentiment-analysis",

model="lxyuan/distilbert-base-multilingual-cased-sentiments-student"

)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [23]:
def sentiment(sentence):

    r = classifier(sentence[:512])[0]

    return r

In [25]:
def score(result):

    label = result["label"].lower()

    conf = result["score"]

    if "positive" in label:

        return 3 + 2*conf

    elif "negative" in label:

        return 3 - 2*conf

    else:

        return 3

In [26]:
records = []

for text in tqdm(df["full_text"]):

    sentences = split_sentences(text)

    for s in sentences:

        topics = detect_topics(s)

        if len(topics)==0:

            continue

        result = sentiment(s)

        value = score(result)

        for t in topics:

            records.append({

                "Topic": t,

                "Sentence": s,

                "Score": value

            })

100%|██████████| 87/87 [01:16<00:00,  1.14it/s]


In [27]:
result_df = pd.DataFrame(records)

result_df.head()

,Topic,Sentence,Score
0,ANC,每一首曲子都像一幅自带环境音的风景画，带着淡淡的怀旧色彩与冬日暖阳般的温度，极具东方美学的留白感,3.000000
1,Connectivity,要说明的是现在的所有专业母带制作系统、广播级制作系统都是异步的TCP/IP连接，若是异步通信...,1.539498
2,Sound,音质好的入耳式耳机应该怎么挑,4.968592
3,Sound,我知道大家都想入手一款听感好的耳机，可商家宣传里的专业术语多数都看不明白，网上的听感评价又都...,4.291940
4,Connectivity,我知道大家都想入手一款听感好的耳机，可商家宣传里的专业术语多数都看不明白，网上的听感评价又都...,4.291940


In [28]:
topic_score = (

result_df

.groupby("Topic")["Score"]

.mean()

.sort_values(ascending=False)

)

print(topic_score)

Topic
Sound           4.334491
Design          4.274590
Comfort         4.250599
Connectivity    4.131689
Battery         4.113122
Price           4.030554
ANC             3.866632
Name: Score, dtype: float64


In [29]:
overall = topic_score.mean()

print("="*40)

print("Sony Product Strength")

print("="*40)

for k,v in topic_score.items():

    print(f"{k:15s} {v:.2f}")

print("-"*40)

print(f"Overall Product Strength: {overall:.2f}")

Sony Product Strength
Sound           4.33
Design          4.27
Comfort         4.25
Connectivity    4.13
Battery         4.11
Price           4.03
ANC             3.87
----------------------------------------
Overall Product Strength: 4.14
